In [1]:
from pyspark.sql import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

# Data Reading

In [2]:
df = spark.read.format("delta")\
    .load("s3://learn-databricks-project-e2e-1-bronze/products")
    
df.show()

+----------+-----------------+-----------+-------+-------+-------------+
|product_id|     product_name|   category|  brand|  price|_rescued_data|
+----------+-----------------+-----------+-------+-------+-------------+
|     P0001|      Clearly Its|     Beauty|   Nike|1868.54|         NULL|
|     P0002| Production Clear|     Beauty|  Apple| 587.13|         NULL|
|     P0003|    Culture Coach|       Home| Revlon|1599.24|         NULL|
|     P0004|    Movement Part|     Sports|     LG| 651.71|         NULL|
|     P0005|        Fact Name|   Clothing|Samsung|1861.78|         NULL|
|     P0006|     Usually Stop|       Toys| Adidas| 936.36|         NULL|
|     P0007|   Reveal Current|     Sports| Adidas|1954.02|         NULL|
|     P0008|   Force Language|     Beauty|   Puma|1251.26|         NULL|
|     P0009|        Stage Leg|   Clothing|Samsung|1247.15|         NULL|
|     P0010|      Leader Then|     Sports|   Sony| 975.53|         NULL|
|     P0011|        Term Rest|Electronics| Adidas|1

In [3]:
df = df.drop("_rescued_data")
df.show()

+----------+-----------------+-----------+-------+-------+
|product_id|     product_name|   category|  brand|  price|
+----------+-----------------+-----------+-------+-------+
|     P0001|      Clearly Its|     Beauty|   Nike|1868.54|
|     P0002| Production Clear|     Beauty|  Apple| 587.13|
|     P0003|    Culture Coach|       Home| Revlon|1599.24|
|     P0004|    Movement Part|     Sports|     LG| 651.71|
|     P0005|        Fact Name|   Clothing|Samsung|1861.78|
|     P0006|     Usually Stop|       Toys| Adidas| 936.36|
|     P0007|   Reveal Current|     Sports| Adidas|1954.02|
|     P0008|   Force Language|     Beauty|   Puma|1251.26|
|     P0009|        Stage Leg|   Clothing|Samsung|1247.15|
|     P0010|      Leader Then|     Sports|   Sony| 975.53|
|     P0011|        Term Rest|Electronics| Adidas|1475.31|
|     P0012|     Theory Wrong|Electronics|  Apple| 624.61|
|     P0013|    Democrat Book|     Sports|   Puma| 384.99|
|     P0014|   Follow Brother|   Clothing|   Dell|   32.

# Functions

In [4]:
df.createOrReplaceTempView("products")

In [5]:
%sql

CREATE OR REPLACE FUNCTION learn_e2e_1.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90


""


In [6]:
%sql

SELECT product_id, price, learn_e2e_1.bronze.discount_func(price) AS discounted_price
FROM products

,product_id,price,discounted_price
0,P0001,1868.54,1681.686
1,P0002,587.13,528.417
2,P0003,1599.24,1439.316
3,P0004,651.71,586.539
4,P0005,1861.78,1675.602
5,P0006,936.36,842.724
6,P0007,1954.02,1758.618
7,P0008,1251.26,1126.134
8,P0009,1247.15,1122.435
9,P0010,975.53,877.977


In [7]:
df.withColumn("discounted_price", expr("learn_e2e_1.bronze.discount_func(price)")).show()

+----------+-----------------+-----------+-------+-------+------------------+
|product_id|     product_name|   category|  brand|  price|  discounted_price|
+----------+-----------------+-----------+-------+-------+------------------+
|     P0001|      Clearly Its|     Beauty|   Nike|1868.54|          1681.686|
|     P0002| Production Clear|     Beauty|  Apple| 587.13|           528.417|
|     P0003|    Culture Coach|       Home| Revlon|1599.24|          1439.316|
|     P0004|    Movement Part|     Sports|     LG| 651.71| 586.5390000000001|
|     P0005|        Fact Name|   Clothing|Samsung|1861.78|          1675.602|
|     P0006|     Usually Stop|       Toys| Adidas| 936.36|           842.724|
|     P0007|   Reveal Current|     Sports| Adidas|1954.02|          1758.618|
|     P0008|   Force Language|     Beauty|   Puma|1251.26|          1126.134|
|     P0009|        Stage Leg|   Clothing|Samsung|1247.15|1122.4350000000002|
|     P0010|      Leader Then|     Sports|   Sony| 975.53|      

In [8]:
%sql

CREATE OR REPLACE FUNCTION learn_e2e_1.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$


""


In [9]:
%sql
SELECT product_id, brand, learn_e2e_1.bronze.upper_func(brand) AS upper_brand
FROM products

,product_id,brand,upper_brand
0,P0001,Nike,NIKE
1,P0002,Apple,APPLE
2,P0003,Revlon,REVLON
3,P0004,LG,LG
4,P0005,Samsung,SAMSUNG
5,P0006,Adidas,ADIDAS
6,P0007,Adidas,ADIDAS
7,P0008,Puma,PUMA
8,P0009,Samsung,SAMSUNG
9,P0010,Sony,SONY


In [10]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("path", "s3://learn-databricks-project-e2e-1-silver/products")\
    .save()

In [11]:
%sql

CREATE TABLE IF NOT EXISTS learn_e2e_1.silver.products
USING DELTA
LOCATION 's3://learn-databricks-project-e2e-1-silver/products'

""
